# Missing-data and small-sample tables
Runs the cohort and eICU steps of `01_jbhi_pipeline.ipynb` with pre-imputation copies retained, then writes aggregate tables to `tables_out/`. Requires MIMIC-IV v2.2 and eICU-CRD v2.0 attached as Kaggle inputs.


In [ ]:
!pip install -q scikit-learn==1.6.1
import sklearn
assert sklearn.__version__ == '1.6.1', sklearn.__version__
print('scikit-learn', sklearn.__version__)


In [ ]:
import os, sys, shutil, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
warnings.filterwarnings('ignore')
PRE = {}
ROOT       = '/kaggle/input/datasets/usamaxai/healthcare/'
MIMIC_HOSP = os.path.join(ROOT, 'MIMIC', 'mimic-iv-3.1', 'hosp')
MIMIC_ICU  = os.path.join(ROOT, 'MIMIC', 'mimic-iv-3.1', 'icu')
EICU       = os.path.join(ROOT, 'EICU', 'eicu-collaborative-research-database-2.0')
OUT        = '/kaggle/working/outputs'
os.makedirs(OUT, exist_ok=True)

N_RUNS  = 30
N_STEPS = 15

VITAL_IDS = {220045:'Heart Rate',
             220181:'Non Invasive Blood Pressure systolic',
             220210:'Respiratory Rate',
             220277:'O2 saturation pulseoxymetry',
             223761:'Temperature Fahrenheit'}
VITAL_NAMES = list(VITAL_IDS.values())

LAB_IDS = {50912:'Creatinine', 51301:'White Blood Cells', 50971:'Potassium',
           50882:'Bicarbonate', 51221:'Hemoglobin', 50931:'Glucose',
           50813:'Lactate'}
VASO_IDS = [221906, 221289, 222315, 221662]
MV_IDS   = [225792, 225794]
SOFA_CHART = {220224:'PaO2', 223835:'FiO2', 220765:'GCS', 220052:'MAP'}
SOFA_LAB   = {51265:'Platelets', 50885:'Bilirubin', 50912:'Creatinine'}
URINE_ID   = 226627


def compute_sofa(cohort):
    stay_ids = cohort['stay_id'].unique()
    hadm_ids = cohort['hadm_id'].unique()
    cv = []
    for c in pd.read_csv(os.path.join(MIMIC_ICU, 'chartevents.csv'),
                         usecols=['stay_id','itemid','valuenum'],
                         iterator=True, chunksize=8_000_000):
        c = c[c['stay_id'].isin(stay_ids) & c['itemid'].isin(SOFA_CHART)].dropna()
        if not c.empty: cv.append(c)
    if cv:
        cv = pd.concat(cv); cv['f'] = cv['itemid'].map(SOFA_CHART)
        cp = cv.groupby(['stay_id','f'])['valuenum'].mean().unstack()
    else:
        cp = pd.DataFrame()
    lv = []
    for c in pd.read_csv(os.path.join(MIMIC_HOSP, 'labevents.csv'),
                         usecols=['hadm_id','itemid','valuenum'],
                         iterator=True, chunksize=8_000_000):
        c = c[c['hadm_id'].isin(hadm_ids) & c['itemid'].isin(SOFA_LAB)].dropna()
        if not c.empty: lv.append(c)
    if lv:
        lv = pd.concat(lv); lv['f'] = lv['itemid'].map(SOFA_LAB)
        lv = lv.merge(cohort[['hadm_id','stay_id']].drop_duplicates(), on='hadm_id')
        lp = lv.groupby(['stay_id','f'])['valuenum'].mean().unstack()
    else:
        lp = pd.DataFrame()
    inp = pd.read_csv(os.path.join(MIMIC_ICU,'inputevents.csv'),
                      usecols=['stay_id','itemid'])
    vaso = pd.DataFrame({'stay_id': list(set(
        inp[inp['itemid'].isin(VASO_IDS)]['stay_id'])), 'vaso_sf':1})
    try:
        oe = pd.read_csv(os.path.join(MIMIC_ICU,'outputevents.csv'),
                         usecols=['stay_id','itemid','value'])
        urine = (oe[oe['itemid']==URINE_ID].groupby('stay_id')['value']
                 .sum().reset_index().rename(columns={'value':'Urine24h'}))
    except Exception:
        urine = pd.DataFrame(columns=['stay_id','Urine24h'])
    sf = cohort[['stay_id']].drop_duplicates()
    for col in ['PaO2','FiO2','GCS','MAP']:
        sf = sf.merge(cp[[col]], on='stay_id', how='left') \
            if (not cp.empty and col in cp.columns) \
            else sf.assign(**{col: np.nan})
    for col in ['Platelets','Bilirubin','Creatinine']:
        sf = sf.merge(lp[[col]], on='stay_id', how='left') \
            if (not lp.empty and col in lp.columns) \
            else sf.assign(**{col: np.nan})
    sf = sf.merge(vaso, on='stay_id', how='left').merge(urine, on='stay_id', how='left')
    sf['vaso_sf'] = sf['vaso_sf'].fillna(0).astype(int)
    def sr(r):
        if pd.isna(r['PaO2']) or pd.isna(r['FiO2']) or r['FiO2']<=0: return np.nan
        v=r['PaO2']/(r['FiO2']/100.)
        return 0 if v>=400 else 1 if v>=300 else 2 if v>=200 else 3 if v>=100 else 4
    def sc(p): return np.nan if pd.isna(p) else (0 if p>=150 else 1 if p>=100 else 2 if p>=50 else 3 if p>=20 else 4)
    def sl(b): return np.nan if pd.isna(b) else (0 if b<1.2 else 1 if b<=1.9 else 2 if b<=5.9 else 3 if b<=11.9 else 4)
    def sca(r):
        m,v=r['MAP'],r['vaso_sf']
        return np.nan if pd.isna(m) else (3 if v==1 else (0 if m>=70 else 1))
    def scns(g):
        if pd.isna(g): return np.nan
        g=int(round(g))
        return 0 if g==15 else 1 if g>=13 else 2 if g>=10 else 3 if g>=6 else 4
    def sren(r):
        c=r['Creatinine']
        return np.nan if pd.isna(c) else (0 if c<1.2 else 1 if c<=1.9 else 2 if c<=3.4 else 3 if c<=4.9 else 4)
    sf['s1']=sf.apply(sr,axis=1); sf['s2']=sf['Platelets'].apply(sc)
    sf['s3']=sf['Bilirubin'].apply(sl); sf['s4']=sf.apply(sca,axis=1)
    sf['s5']=sf['GCS'].apply(scns); sf['s6']=sf.apply(sren,axis=1)
    comp=['s1','s2','s3','s4','s5','s6']
    PRE['sofa_raw']=sf[['stay_id']+comp].copy()  # pre-imputation copy for missingness tables
    for c in comp:
        med=sf[c].median()
        sf[c]=sf[c].fillna(med if not pd.isna(med) else 1)
    sf['SOFA_score']=sf[comp].sum(axis=1)
    return sf[['stay_id','SOFA_score']].copy()


def extract_hourly_vitals(stay_ids_ordered, intime_map):
    sid_set = set(stay_ids_ordered)
    sid_idx = {s:i for i,s in enumerate(stay_ids_ordered)}
    vid_idx = {v:i for i,v in enumerate(VITAL_NAMES)}
    chunks  = []
    for c in pd.read_csv(os.path.join(MIMIC_ICU, 'chartevents.csv'),
                         usecols=['stay_id','itemid','charttime','valuenum'],
                         iterator=True, chunksize=8_000_000):
        c = c[c['stay_id'].isin(sid_set) & c['itemid'].isin(VITAL_IDS)].dropna()
        if not c.empty: chunks.append(c)
    ce = pd.concat(chunks)
    ce['charttime'] = pd.to_datetime(ce['charttime'])
    it = pd.DataFrame({'stay_id': list(intime_map.keys()),
                       'intime':  pd.to_datetime(list(intime_map.values()))})
    ce = ce.merge(it, on='stay_id')
    ce['hour'] = ((ce['charttime']-ce['intime']).dt.total_seconds()/3600).astype(int)
    ce = ce[(ce['hour']>=0)&(ce['hour']<24)]
    ce['vname'] = ce['itemid'].map(VITAL_IDS)
    N = len(stay_ids_ordered)
    X = np.full((N,24,5), np.nan, dtype=np.float32)
    grp = ce.groupby(['stay_id','hour','vname'])['valuenum'].mean()
    for (sid,hr,vn),val in grp.items():
        if sid in sid_idx and vn in vid_idx:
            X[sid_idx[sid], hr, vid_idx[vn]] = val
    PRE['vit_obs']=~np.isnan(X)  # pre-imputation copy for missingness tables
    for i in range(N):
        for j in range(5):
            s = pd.Series(X[i,:,j]).ffill().bfill()
            X[i,:,j] = s.values
    for j in range(5):
        cm = float(np.nanmean(X[:,:,j]))
        X[:,:,j] = np.where(np.isnan(X[:,:,j]), cm, X[:,:,j])
    means = X.mean(axis=(0,1), keepdims=True)
    stds  = X.std(axis=(0,1),  keepdims=True)
    stds[stds==0] = 1.0
    X_norm = ((X - means) / stds).astype(np.float32)
    agg = np.zeros((N,25), dtype=np.float32)
    for j in range(5):
        agg[:,j*5+0] = X[:,:,j].mean(axis=1)
        agg[:,j*5+1] = X[:,:,j].min(axis=1)
        agg[:,j*5+2] = X[:,:,j].max(axis=1)
        agg[:,j*5+3] = X[:,0,j]
        agg[:,j*5+4] = X[:,23,j]
    return X_norm, agg, means.squeeze(), stds.squeeze()


def preprocess_mimic():
    print('--- MIMIC Preprocessing ---')
    patients  = pd.read_csv(os.path.join(MIMIC_HOSP,'patients.csv'),
                            usecols=['subject_id','gender','anchor_age','dod'])
    admissions= pd.read_csv(os.path.join(MIMIC_HOSP,'admissions.csv'),
                            usecols=['subject_id','hadm_id','race','dischtime'])
    icustays  = pd.read_csv(os.path.join(MIMIC_ICU,'icustays.csv'),
                            usecols=['subject_id','hadm_id','stay_id','intime'])
    diagnoses = pd.read_csv(os.path.join(MIMIC_HOSP,'diagnoses_icd.csv'))
    sep_ids   = diagnoses[diagnoses['icd_code'].isin(
        ['99591','99592','78552'])]['hadm_id'].unique()
    df = (admissions[admissions['hadm_id'].isin(sep_ids)]
          .merge(patients,on='subject_id').merge(icustays,on=['subject_id','hadm_id']))
    df = (df[df['anchor_age']>18].sort_values(['subject_id','intime'])
          .drop_duplicates('subject_id',keep='first'))
    df['mortality_in_hospital']=(
        pd.to_datetime(df['dod']).notna() &
        (pd.to_datetime(df['dod'])<=pd.to_datetime(df['dischtime']))).astype(int)
    def eth(r):
        r=str(r).upper()
        if 'WHITE' in r: return 'White'
        if 'BLACK' in r: return 'Black'
        if 'ASIAN' in r or 'HISPANIC' in r: return 'Asian-Hispanic'
        return 'Other'
    df['ethnicity']  = df['race'].apply(eth)
    df['age_group']  = pd.cut(df['anchor_age'],bins=[18,65,150],
                               labels=['Under65','Over65'],right=False).astype(str)
    df['intersectional_group']=(df['ethnicity']+'_'+df['gender']+'_'+df['age_group'])
    cohort = df[['subject_id','hadm_id','stay_id','gender','anchor_age',
                 'ethnicity','age_group','intersectional_group',
                 'mortality_in_hospital']].copy()
    stay_ids = cohort['stay_id'].unique()
    hadm_ids = cohort['hadm_id'].unique()
    sofa_df  = compute_sofa(cohort)
    vc = [c[c['stay_id'].isin(stay_ids)&c['itemid'].isin(VITAL_IDS)].dropna()
          for c in pd.read_csv(os.path.join(MIMIC_ICU,'chartevents.csv'),
                               usecols=['stay_id','itemid','valuenum'],
                               iterator=True, chunksize=8_000_000)]
    vit = (pd.concat(vc).groupby(['stay_id','itemid'])['valuenum'].mean()
           .reset_index()
           .pivot(index='stay_id', columns='itemid', values='valuenum')
           .rename(columns=VITAL_IDS))
    lc = [c[c['hadm_id'].isin(hadm_ids)&c['itemid'].isin(LAB_IDS)].dropna()
          for c in pd.read_csv(os.path.join(MIMIC_HOSP,'labevents.csv'),
                               usecols=['hadm_id','itemid','valuenum'],
                               iterator=True, chunksize=8_000_000)]
    labs_df = pd.concat(lc)
    labs_ws = labs_df.merge(cohort[['hadm_id','stay_id']].drop_duplicates(),
                             on='hadm_id',how='left').drop_duplicates()
    labs    = (labs_ws.groupby(['stay_id','itemid'])['valuenum'].mean()
               .reset_index()
               .pivot(index='stay_id', columns='itemid', values='valuenum')
               .rename(columns=LAB_IDS))
    diag_agg= (diagnoses.groupby('hadm_id')['icd_code'].nunique()
               .reset_index().rename(columns={'icd_code':'comorbidity_index'}))
    try:
        inp = pd.read_csv(os.path.join(MIMIC_ICU,'inputevents.csv'),
                          usecols=['stay_id','itemid'])
        vp  = pd.DataFrame({'stay_id':list(set(inp[inp['itemid'].isin(VASO_IDS)]['stay_id'])),
                            'vasopressor_use':1})
    except Exception:
        vp  = pd.DataFrame(columns=['stay_id','vasopressor_use'])
    try:
        proc= pd.read_csv(os.path.join(MIMIC_ICU,'procedureevents.csv'),
                          usecols=['stay_id','itemid'])
        mv  = pd.DataFrame({'stay_id':list(set(proc[proc['itemid'].isin(MV_IDS)]['stay_id'])),
                            'mechanical_ventilation':1})
    except Exception:
        mv  = pd.DataFrame(columns=['stay_id','mechanical_ventilation'])
    final = (cohort.merge(vit,on='stay_id',how='left')
             .merge(labs,on='stay_id',how='left')
             .merge(diag_agg,on='hadm_id',how='left')
             .merge(vp,on='stay_id',how='left')
             .merge(mv,on='stay_id',how='left')
             .merge(sofa_df,on='stay_id',how='left'))
    final['vasopressor_use']       =final['vasopressor_use'].fillna(0).astype(int)
    final['mechanical_ventilation']=final['mechanical_ventilation'].fillna(0).astype(int)
    intime_map = (icustays[icustays['stay_id'].isin(stay_ids)]
                  .set_index('stay_id')['intime'].to_dict())
    final = final.drop(columns=['subject_id','hadm_id'])
    num_c = final.select_dtypes(include=np.number).columns.difference(
        ['mortality_in_hospital','stay_id'])
    PRE['mimic_pre']=final.copy()  # pre-imputation copy for missingness tables
    final[num_c]=SimpleImputer(strategy='median').fit_transform(final[num_c])
    final.to_csv(os.path.join(OUT,'sepsis_cohort_17feat.csv'),index=False)
    pd.DataFrame({'stay_id':list(intime_map.keys()),
                  'intime': list(intime_map.values())}).to_csv(
        os.path.join(OUT,'intime_map.csv'),index=False)
    print(f'Preprocessed: {final.shape}')
    return final, intime_map


def bootstrap_auc(yt,yp,n=1000,seed=42):
    rng=np.random.RandomState(seed); aucs=[]; N=len(yt)
    for _ in range(n):
        idx=rng.randint(0,N,N); a,b=np.array(yt)[idx],np.array(yp)[idx]
        if len(np.unique(a))<2: continue
        aucs.append(roc_auc_score(a,b))
    return (round(float(np.mean(aucs)),4),
            round(float(np.percentile(aucs,2.5)),4),
            round(float(np.percentile(aucs,97.5)),4))


def fairness_metrics(yt,yp,grp):
    yt=pd.Series(yt).reset_index(drop=True)
    yp=pd.Series(yp).reset_index(drop=True)
    grp=pd.Series(grp).reset_index(drop=True)
    rows=[]
    for g in grp.unique():
        m=grp==g; ytg=yt[m].values; ypg=yp[m].values
        if len(ytg)<2 or len(np.unique(ytg))<2:
            tn,fp,fn,tp=0,0,0,int(ytg.sum())
        else:
            tn,fp,fn,tp=confusion_matrix(ytg,ypg,labels=[0,1]).ravel()
        tpr=tp/(tp+fn) if (tp+fn)>0 else 0.
        fpr=fp/(fp+tn) if (fp+tn)>0 else 0.
        fo =fn/(fn+tn) if (fn+tn)>0 else 0.
        rows.append({'group':g,'TPR':tpr,'FPR':fpr,'FOR':fo,
                     'n':int(m.sum()),'n_pos':int(ytg.sum())})
    dfm=pd.DataFrame(rows).set_index('group')
    gaps={'tpr_gap':float(dfm['TPR'].max()-dfm['TPR'].min()),
          'tpr_std':float(dfm['TPR'].std()),
          'fpr_gap':float(dfm['FPR'].max()-dfm['FPR'].min()),
          'for_gap':float(dfm['FOR'].max()-dfm['FOR'].min())}
    return dfm, gaps


def eicu_fairness(df_mimic):
    print('\n--- eICU Cross-Institutional Fairness ---')
    try:
        pat = pd.read_csv(os.path.join(EICU,'patient.csv'),
                          usecols=['patientunitstayid','gender','age',
                                   'hospitalid','unitdischargestatus','ethnicity'])
    except Exception as e:
        print(f'eICU unavailable: {e}'); return None
    pat['age_num'] = pd.to_numeric(
        pat['age'].astype(str).str.replace('>',''), errors='coerce')
    pat = pat.dropna(subset=['age_num'])
    pat = pat[pat['age_num']>18]
    def mape(e):
        e = str(e).upper()
        if 'CAUCASIAN' in e: return 'White'
        if 'AFRICAN' in e: return 'Black'
        if 'ASIAN' in e or 'HISPANIC' in e: return 'Asian-Hispanic'
        return 'Other'
    pat['ethnicity'] = pat['ethnicity'].apply(mape)
    pat['age_group'] = np.where(pat['age_num']<65,'Under65','Over65')
    pat['gender_s'] = pat['gender'].str.upper().map(
        {'MALE':'M','FEMALE':'F'}).fillna('F')
    pat['intersectional_group'] = (
        pat['ethnicity']+'_'+pat['gender_s']+'_'+pat['age_group'])
    pat['mortality'] = (
        pat['unitdischargestatus'].str.upper()=='EXPIRED').astype(int)
    pat.rename(columns={'patientunitstayid':'stay_id',
                        'age_num':'anchor_age'}, inplace=True)

    try:
        vit_chunks = pd.read_csv(
            os.path.join(EICU,'vitalPeriodic.csv/vitalPeriodic.csv'),
            usecols=['patientunitstayid','heartrate','systemicsystolic',
                     'respiration','sao2','temperature'],
            iterator=True, chunksize=2_000_000)
        vit_df = pd.concat(vit_chunks)
        vit_df.rename(columns={'patientunitstayid':'stay_id',
                                'heartrate':'Heart Rate',
                                'systemicsystolic':'Non Invasive Blood Pressure systolic',
                                'respiration':'Respiratory Rate',
                                'sao2':'O2 saturation pulseoxymetry',
                                'temperature':'Temperature Fahrenheit'},
                       inplace=True)
        vit_agg = vit_df.groupby('stay_id')[VITAL_NAMES].mean().reset_index()
        pat = pat.merge(vit_agg, on='stay_id', how='left')
    except Exception as e:
        print(f'  vitals unavailable: {e}')
        for v in VITAL_NAMES:
            pat[v] = np.nan

    try:
        lab = pd.read_csv(os.path.join(EICU,'lab.csv'),
                          usecols=['patientunitstayid','labresultoffset',
                                   'labname','labresult'])
        lab.rename(columns={'patientunitstayid':'stay_id'}, inplace=True)
        lab = lab[lab['labresultoffset'].between(0,1440)]
        lm = {'creatinine':'Creatinine','wbc':'White Blood Cells',
              'potassium':'Potassium','bicarbonate':'Bicarbonate',
              'hgb':'Hemoglobin','glucose':'Glucose','lactate':'Lactate'}
        lab['labname'] = lab['labname'].str.lower()
        lab_p = (lab[lab['labname'].isin(lm)]
                 .assign(feat=lambda d: d['labname'].map(lm))
                 .groupby(['stay_id','feat'])['labresult'].mean()
                 .reset_index()
                 .pivot(index='stay_id', columns='feat', values='labresult')
                 .reset_index())
        pat = pat.merge(lab_p, on='stay_id', how='left')
    except Exception as e:
        print(f'  labs unavailable: {e}')

    common_features = VITAL_NAMES + list(LAB_IDS.values()) + ['anchor_age']
    for c in common_features:
        if c not in pat.columns:
            pat[c] = 0.0

    PRE['eicu_pre']=pat.copy()  # pre-imputation copy for missingness tables
    X_eicu = pat[common_features].fillna(0.0).astype(np.float32)
    y_eicu = pat['mortality'].values
    g_eicu = pat['intersectional_group'].values

    X_mimic = df_mimic[common_features].fillna(0.0).astype(np.float32)
    y_mimic = df_mimic['mortality_in_hospital'].values
    sc = StandardScaler()
    Xm = sc.fit_transform(X_mimic)
    Xe = sc.transform(X_eicu)
    rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
    rf.fit(Xm, y_mimic)
    pb_e = rf.predict_proba(Xe)[:,1]
    pr_e = (pb_e>0.5).astype(int)
    PRE['eicu_pred']=pd.DataFrame({'y':pat['mortality'].values,'pred':pr_e,'group':pat['intersectional_group'].values,'hospitalid':pat['hospitalid'].values})  # pre-imputation copy for missingness tables
    auc_e, ci_lo, ci_hi = bootstrap_auc(y_eicu, pb_e)
    pd.DataFrame({'AUC':[auc_e],'CI_lo':[ci_lo],'CI_hi':[ci_hi]}).to_csv(
        os.path.join(OUT,'eicu_rf_auc.csv'), index=False)
    print(f'eICU RF AUC={auc_e} (95% CI: {ci_lo}-{ci_hi})')

    dfm_e, gaps_e = fairness_metrics(y_eicu, pr_e, g_eicu)
    dfm_e.round(4).to_csv(
        os.path.join(OUT,'table_eicu_fairness_overall.csv'))
    print(f'eICU TPR gap={gaps_e["tpr_gap"]:.4f} '
          f'FPR gap={gaps_e["fpr_gap"]:.4f} '
          f'FOR gap={gaps_e["for_gap"]:.4f}')

    hosp = []
    pat_with_pred = pat.copy()
    pat_with_pred['_pred'] = pr_e
    for hid, grp in pat_with_pred.groupby('hospitalid'):
        if len(grp) < 30: continue
        _, gps = fairness_metrics(
            grp['mortality'].values,
            grp['_pred'].values,
            grp['intersectional_group'].values)
        hosp.append({'hospitalid':hid,'n':len(grp),
                     'tpr_gap':round(gps['tpr_gap'],4),
                     'fpr_gap':round(gps['fpr_gap'],4),
                     'for_gap':round(gps['for_gap'],4)})
    n_hospitals = 0
    if hosp:
        hdf = pd.DataFrame(hosp).set_index('hospitalid').sort_values(
            'tpr_gap', ascending=False)
        hdf.to_csv(os.path.join(OUT,'table_eicu_fairness_hospital.csv'))
        fig, axes = plt.subplots(1,3, figsize=(14,5))
        for ax, col, lab in zip(axes,
                                  ['tpr_gap','fpr_gap','for_gap'],
                                  ['TPR Gap','FPR Gap','FOR Gap']):
            if col in hdf.columns and hdf[col].notna().any():
                ax.boxplot(hdf[col].dropna(), labels=[lab])
                ax.set_title(f'Cross-Institutional {lab}')
                ax.set_ylabel(f'Max-Min {lab}')
                ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(os.path.join(OUT,'figure_eicu_hosp_fairness.png'),
                    dpi=300)
        plt.close()
        n_hospitals = len(hdf)
        print(f'Per-hospital fairness: {n_hospitals} hospitals analyzed')
    return {'rf_auc':auc_e,'ci_lo':ci_lo,'ci_hi':ci_hi,
            'tpr_gap':gaps_e['tpr_gap'],'fpr_gap':gaps_e['fpr_gap'],
            'for_gap':gaps_e['for_gap'], 'n_hospitals':n_hospitals}


In [ ]:
# Cell 2: AE missingness and small-sample functions
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import betaln
from scipy.stats import chi2_contingency, spearmanr
Z = 1.959964


def wilson(k, n):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    den = 1 + Z * Z / n
    c = (p + Z * Z / (2 * n)) / den
    h = Z * np.sqrt(p * (1 - p) / n + Z * Z / (4 * n * n)) / den
    return max(0.0, c - h), min(1.0, c + h)


def missingness_tables(df, preds, label, out, hosp_col=None):
    overall = pd.DataFrame({"predictor": preds,
                            "n_missing": [int(df[p].isna().sum()) for p in preds],
                            "pct_missing": [round(100 * df[p].isna().mean(), 2) for p in preds]})
    by_g = df.groupby("intersectional_group")[preds].apply(lambda x: x.isna().mean() * 100).round(2)
    by_g.to_csv(os.path.join(out, f"{label}_missing_by_subgroup.csv"))
    pvals, lo, hi = [], [], []
    for p in preds:
        tab = pd.crosstab(df["intersectional_group"], df[p].isna())
        pvals.append(chi2_contingency(tab)[1] if tab.shape[1] == 2 else np.nan)
        lo.append(by_g[p].min()); hi.append(by_g[p].max())
    overall["subgroup_min_pct"], overall["subgroup_max_pct"] = lo, hi
    overall["chi2_p_subgroup"] = np.round(pvals, 4)
    if hosp_col:
        by_h = df.groupby(hosp_col)[preds].apply(lambda x: x.isna().mean() * 100)
        by_h.round(2).to_csv(os.path.join(out, f"{label}_missing_by_hospital.csv"))
        overall["hosp_median_pct"] = by_h.median().round(2).values
        overall["hosp_q1_pct"] = by_h.quantile(.25).round(2).values
        overall["hosp_q3_pct"] = by_h.quantile(.75).round(2).values
        overall["hosp_all_missing_n"] = (by_h == 100).sum().values
        overall["n_hospitals"] = len(by_h)
    overall.to_csv(os.path.join(out, f"{label}_missing_overall.csv"), index=False)
    return overall


def subgroup_counts(y, pred, g):
    d = pd.DataFrame({"y": y, "pred": pred, "g": g})
    rows = []
    for k, s in d.groupby("g"):
        tp = int(((s.y == 1) & (s.pred == 1)).sum()); fn = int(((s.y == 1) & (s.pred == 0)).sum())
        fp = int(((s.y == 0) & (s.pred == 1)).sum()); tn = int(((s.y == 0) & (s.pred == 0)).sum())
        r = dict(group=k, n=len(s), events=tp + fn, tp=tp, fn=fn, fp=fp, tn=tn)
        for name, num, den in (("TPR", tp, tp + fn), ("FPR", fp, fp + tn), ("FOR", fn, fn + tn)):
            r[name] = num / den if den else np.nan
            r[name + "_lo"], r[name + "_hi"] = wilson(num, den)
            r[name + "_den"] = den
        rows.append(r)
    return pd.DataFrame(rows).sort_values("n", ascending=False)


def fit_beta_prior(x, n):
    """Beta-binomial marginal likelihood; returns (a, b)."""
    x, n = np.asarray(x, float), np.asarray(n, float)
    keep = n > 0
    x, n = x[keep], n[keep]
    if len(x) < 3:
        p = x.sum() / max(n.sum(), 1)
        return p * 10, (1 - p) * 10

    def nll(t):
        a, b = np.exp(t)
        return -np.sum(betaln(x + a, n - x + b) - betaln(a, b))
    p0 = np.clip(x.sum() / n.sum(), .02, .98)
    res = minimize(nll, np.log([p0 * 10, (1 - p0) * 10]), method="Nelder-Mead")
    return tuple(np.exp(res.x))


def eicu_hospital_block(path, out, min_cell, min_hosp_n, lines):
    d = pd.read_csv(path)
    a_tpr = b_tpr = None
    cells = []
    for h, s in d.groupby("hospitalid"):
        if len(s) < min_hosp_n:
            continue
        t = subgroup_counts(s.y.values, s.pred.values, s.group.values)
        t["hospitalid"], t["hosp_n"] = h, len(s)
        cells.append(t)
    cells = pd.concat(cells, ignore_index=True)
    a_tpr, b_tpr = fit_beta_prior(cells.tp, cells.TPR_den)
    a_for, b_for = fit_beta_prior(cells.fn, cells.FOR_den)
    rows = []
    for h, t in cells.groupby("hospitalid"):
        v = t[t.TPR_den > 0]; vf = t[t.FOR_den > 0]
        r = dict(hospitalid=h, n=int(t.hosp_n.iloc[0]), events=int(t.events.sum()),
                 cells_tpr_ge_min=int((v.TPR_den >= min_cell).sum()),
                 tpr_gap_raw=v.TPR.max() - v.TPR.min() if len(v) > 1 else np.nan,
                 for_gap_raw=vf.FOR.max() - vf.FOR.min() if len(vf) > 1 else np.nan)
        vv = v[v.TPR_den >= min_cell]
        r["tpr_gap_mincell"] = vv.TPR.max() - vv.TPR.min() if len(vv) > 1 else np.nan
        pt = (v.tp + a_tpr) / (v.TPR_den + a_tpr + b_tpr)
        pf = (vf.fn + a_for) / (vf.FOR_den + a_for + b_for)
        r["tpr_gap_eb"] = pt.max() - pt.min() if len(pt) > 1 else np.nan
        r["for_gap_eb"] = pf.max() - pf.min() if len(pf) > 1 else np.nan
        rows.append(r)
    H = pd.DataFrame(rows)
    H["size_quartile"] = pd.qcut(H.n, 4, labels=["Q1", "Q2", "Q3", "Q4"])
    H.round(4).to_csv(os.path.join(out, "eicu_hospital_gaps_smallsample.csv"), index=False)
    Q = H.groupby("size_quartile", observed=True).agg(
        hospitals=("n", "size"), median_n=("n", "median"), median_events=("events", "median"),
        tpr_raw=("tpr_gap_raw", "median"), tpr_eb=("tpr_gap_eb", "median"),
        tpr_mincell=("tpr_gap_mincell", "median"), hosp_with_2plus_eligible_cells=("cells_tpr_ge_min", lambda x: int((x >= 2).sum())),
        for_raw=("for_gap_raw", "median"), for_eb=("for_gap_eb", "median")).round(3)
    Q.to_csv(os.path.join(out, "eicu_quartiles_smallsample.csv"))
    lines.append(f"\n[eICU per-hospital] hospitals={len(H)}, stays={H.n.sum()}, events={H.events.sum()}")
    lines.append(f"  hospitals with >=2 subgroups having >={min_cell} events: {int((H.cells_tpr_ge_min >= 2).sum())}")
    for c in ("tpr_gap_raw", "tpr_gap_eb", "for_gap_raw", "for_gap_eb"):
        s = H[[c, "n"]].dropna()
        rho, p = spearmanr(s.n, s[c])
        lines.append(f"  Spearman(n, {c}) rho={rho:.3f} p={p:.4f}")
    lines.append(Q.to_string())


In [ ]:
# Cell 3: run (MIMIC preprocessing + hourly vitals + eICU step only; no simulation)
OUT = '/kaggle/working/tmp_out'; os.makedirs(OUT, exist_ok=True)
R_OUT = '/kaggle/working/tables_out'; os.makedirs(R_OUT, exist_ok=True)
lines = []

df, intime_map = preprocess_mimic()
lines.append(f"MIMIC cohort n={len(df)} mortality={df['mortality_in_hospital'].mean():.4f} (published: 5106, 0.2793)")
_ = extract_hourly_vitals(df['stay_id'].tolist(), intime_map)

pre = PRE['mimic_pre'].copy()
labs = list(LAB_IDS.values())
obs = PRE['vit_obs']                      # N x 24 x 5 observed mask, same order as df
vit_none = pd.DataFrame(~obs.any(axis=1), columns=[f'{v} (24h)' for v in VITAL_NAMES])
vit_none = vit_none.replace({True: np.nan, False: 1.0})
sofa = PRE['sofa_raw'].rename(columns={'s1':'SOFA respiratory','s2':'SOFA coagulation','s3':'SOFA liver',
                                      's4':'SOFA cardiovascular','s5':'SOFA CNS','s6':'SOFA renal'})
m = (pre[['stay_id','intersectional_group','mortality_in_hospital','anchor_age','comorbidity_index'] + labs]
     .reset_index(drop=True))
m = pd.concat([m, vit_none], axis=1).merge(sofa, on='stay_id', how='left')
preds = ['anchor_age'] + [f'{v} (24h)' for v in VITAL_NAMES] + labs + list(sofa.columns.drop('stay_id')) + ['comorbidity_index']
o = missingness_tables(m, preds, 'mimic', R_OUT)
lines.append('[MIMIC-IV missingness: % stays with no value before imputation]\n' + o.to_string(index=False))
hrs = pd.DataFrame({v: 100 * (1 - obs[:, :, j].mean(axis=1)) for j, v in enumerate(VITAL_NAMES)})
hrs.describe().round(2).to_csv(os.path.join(R_OUT, 'mimic_vitals_pct_hours_unobserved.csv'))
lines.append('[MIMIC-IV % of the 24 hourly slots unobserved per stay]\n' + hrs.describe().round(2).to_string())

es = eicu_fairness(df)               # prints AUC; published 0.7431 (0.7375-0.7485)
e = PRE['eicu_pre']; ep = PRE['eicu_pred']
lines.append(f"\neICU analysed stays={len(e)}, deaths={int(e['mortality'].sum())}, hospitals={e['hospitalid'].nunique()}, AUC={es['rf_auc']}")
cf = VITAL_NAMES + list(LAB_IDS.values()) + ['anchor_age']
o2 = missingness_tables(e.rename(columns={'mortality':'mortality_in_hospital'}), cf, 'eicu', R_OUT, hosp_col='hospitalid')
lines.append('[eICU missingness]\n' + o2.to_string(index=False))
ep.to_csv('/kaggle/working/eicu_preds_patientlevel.csv', index=False)   # patient-level file is kept in the session only
eicu_hospital_block('/kaggle/working/eicu_preds_patientlevel.csv', R_OUT, 10, 30, lines)
g = ep.groupby('group').agg(n=('y','size'), deaths=('y','sum')).reset_index()
g.to_csv(os.path.join(R_OUT, 'eicu_subgroup_counts.csv'), index=False)

open(os.path.join(R_OUT, 'summary.txt'), 'w').write('\n'.join(lines))
print('\n'.join(lines))
shutil.make_archive('/kaggle/working/tables_out', 'zip', R_OUT)
print('\nDone. Download /kaggle/working/tables_out.zip (aggregate tables only)')
